# Week 2 Day 3 — LangGraph: Stateful, Multi-Step & Cyclical Agent Workflows

Moves from Day 2's `AgentExecutor`/`create_agent` (a single reasoning loop) to **LangGraph**,
which models the workflow as an explicit graph of nodes and edges — with branching, a
self-correction loop, a human-in-the-loop approval gate, and persistence/time-travel.

**Workflow chosen:** a *budget laptop recommendation assistant* that plans, retrieves prices
(reusing Day 2's `get_product_price` tool and `products.json`), drafts a recommendation,
critiques its own draft, loops back to revise if the critique score is too low, then pauses
for human approval before "sending" the recommendation email to the client.


## Setup

Run once in a terminal (or a `!pip install` cell):

```
pip install -U langchain langchain-core langchain-google-genai langgraph python-dotenv pydantic
```

Copy the same `.env` (with `GEMINI_API_KEY`) used in Day 1/Day 2 into this folder.


In [ ]:
# !pip install -q -U langchain langchain-core langchain-google-genai langgraph python-dotenv pydantic


In [36]:
import os
import json
from typing import TypedDict, Optional

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise RuntimeError("GEMINI_API_KEY not found in .env — reuse the same key as Day 1/Day 2.")

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", temperature=0)
def get_text(resp):
    """Extract clean text from an LLM response, whether content is a plain
    string (older models) or a list of structured content blocks (newer models
    like gemini-3.5-flash-lite)."""
    content = resp.content
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "".join(block.get("text", "") for block in content if isinstance(block, dict))
    return str(content)
print("LLM ready:", llm.model)


LLM ready: gemini-3.5-flash-lite


## Task 1: Graph Concepts & State Design

### Core building blocks

- **`StateGraph`** — the graph builder. You register nodes and edges on it, then `.compile()` it
  into a runnable graph (similar role to `AgentExecutor`, but you draw the control flow yourself
  instead of it being an implicit reason/act/observe loop).
- **Nodes** — plain Python functions `(state) -> partial_state_update`. Each node reads the shared
  `State` and returns a dict of the fields it changed; LangGraph merges that into the full state.
- **Edges** — fixed transitions (`add_edge("a", "b")`: always go from `a` to `b`).
- **Conditional edges** — `add_conditional_edges("a", router_fn, {"key": "b", ...})`: after node
  `a` runs, `router_fn(state)` inspects the state and returns a key that decides which node runs
  next. This is what makes branching *and* loops (a "key" can route back to an earlier node)
  possible.
- **The shared `State` object** — a single `TypedDict` (or Pydantic model) that every node reads
  from and writes to. It's the explicit replacement for Day 2's ad-hoc `scratchpad` dict — every
  field that matters to the workflow is declared up front.

### State schema for this workflow


In [37]:
class RecommendationState(TypedDict):
    question: str                  # the user's original question
    plan: str                      # short plan produced by the "plan" node
    retrieved: str                 # raw tool output from "retrieve"
    draft: str                     # current draft recommendation
    critique_feedback: str         # feedback from the last critique pass
    score: int                     # critique's 0-100 quality score for the current draft
    retries: int                   # how many revise->generate loops have run so far
    max_retries: int               # hard cap to prevent infinite cycles
    approved: Optional[bool]       # human's decision at the approval gate (None until decided)
    final: str                     # the final output (email-sent text or cancellation note)


### Graph diagram (drawn before coding it)

```
        START
          |
          v
        plan
          |
          v
       retrieve
          |
          v
    +-> generate
    |     |
    |     v
    |  critique
    |     |
    |     v
    |  [route_critique]
    |   /            \
    |  / score<80 &    \ score>=80
    | / retries<max     \ or retries>=max
    | v                  v
    +revise          send_email   <-- interrupt_before pauses HERE for human approval
                          |
                          v
                         END
```

The Mermaid version below renders directly in VS Code / Jupyter / GitHub markdown previews:

```mermaid
flowchart TD
    START([START]) --> plan
    plan --> retrieve
    retrieve --> generate
    generate --> critique
    critique -->|score >= 80 OR retries >= max| send_email
    critique -->|score < 80 AND retries < max| revise
    revise --> generate
    send_email -->|interrupt_before: pauses for human approval| END([END])
```


## Task 2: Build a Linear Graph (plan -> retrieve -> generate -> critique)

First build and run the graph as a straight line (no loop-back yet) to verify state updates
correctly after each node, before adding the conditional cycle in Task 3.


In [38]:
# Reused from Day 2: the local JSON "database" and the get_product_price tool
PRODUCTS_DB_PATH = "products.json"
products_data = {
    "laptop a": {"price_usd": 650, "specs": "8GB RAM, 256GB SSD, 14-inch"},
    "laptop b": {"price_usd": 950, "specs": "16GB RAM, 512GB SSD, 15-inch"},
    "laptop c": {"price_usd": 1200, "specs": "32GB RAM, 1TB SSD, 16-inch"},
}
with open(PRODUCTS_DB_PATH, "w") as f:
    json.dump(products_data, f, indent=2)

@tool
def get_product_price(product_name: str) -> str:
    """Look up the price and specs of a product from the local product database."""
    with open(PRODUCTS_DB_PATH) as f:
        db = json.load(f)
    key = product_name.strip().lower()
    if key not in db:
        return f"ERROR: no product named '{product_name}' in the database."
    info = db[key]
    return f"{product_name}: ${info['price_usd']} ({info['specs']})"

print(get_product_price.invoke({"product_name": "Laptop A"}))


Laptop A: $650 (8GB RAM, 256GB SSD, 14-inch)


In [40]:
def plan_node(state: RecommendationState) -> dict:
    """Node 1: ask the LLM for a one-sentence plan of which products to compare."""
    resp = llm.invoke(
        f"User question: {state['question']}\n"
        "In one short sentence, state a plan for which specific products to look up "
        "to answer this (name them if the question does)."
    )
    plan_text = get_text(resp)
    print(f"[plan] {plan_text}")
    return {"plan": plan_text}

def retrieve_node(state: RecommendationState) -> dict:
    """Node 2: deterministically look up Laptop A and Laptop B prices (the two
    products this demo workflow compares) using Day 2's tool."""
    a = get_product_price.invoke({"product_name": "Laptop A"})
    b = get_product_price.invoke({"product_name": "Laptop B"})
    retrieved = f"{a}\n{b}"
    print(f"[retrieve] {retrieved}")
    return {"retrieved": retrieved}

def generate_node(state: RecommendationState) -> dict:
    """Node 3: draft a recommendation using the retrieved prices and, on a retry,
    the previous critique feedback."""
    feedback_note = f"\nPrevious feedback to address: {state['critique_feedback']}" if state.get("critique_feedback") else ""
    resp = llm.invoke(
        f"Question: {state['question']}\n"
        f"Plan: {state['plan']}\n"
        f"Retrieved data:\n{state['retrieved']}{feedback_note}\n\n"
        "Write a short (2-3 sentence) recommendation for a budget-conscious client, "
        "explicitly mentioning both prices and your reasoning."
    )
    draft_text = get_text(resp)
    print(f"[generate] {draft_text}")
    return {"draft": draft_text}


def critique_node(state: RecommendationState) -> dict:
    """Node 4: score the draft 0-100 and give one sentence of feedback."""

    class Critique(BaseModel):
        score: int = Field(description="Quality score from 0 to 100")
        feedback: str = Field(description="One sentence of feedback for improvement")

    structured_llm = llm.with_structured_output(Critique)
    result = structured_llm.invoke(
        f"Draft recommendation: {state['draft']}\n\n"
        "Score this draft 0-100 on whether it clearly mentions BOTH prices and gives "
        "a clear reason for a budget-conscious client. Give one sentence of feedback."
    )
    print(f"[critique] score={result.score} feedback={result.feedback}")
    return {"score": result.score, "critique_feedback": result.feedback}


In [41]:
# Compile a straight-line graph first (no conditional edges / cycle yet) to verify
# state flows correctly node by node.
linear_builder = StateGraph(RecommendationState)
linear_builder.add_node("plan", plan_node)
linear_builder.add_node("retrieve", retrieve_node)
linear_builder.add_node("generate", generate_node)
linear_builder.add_node("critique", critique_node)
linear_builder.add_edge(START, "plan")
linear_builder.add_edge("plan", "retrieve")
linear_builder.add_edge("retrieve", "generate")
linear_builder.add_edge("generate", "critique")
linear_builder.add_edge("critique", END)

linear_graph = linear_builder.compile()

initial_state = {
    "question": "Which laptop should a budget-conscious client buy, Laptop A or Laptop B?",
    "plan": "", "retrieved": "", "draft": "", "critique_feedback": "",
    "score": 0, "retries": 0, "max_retries": 2, "approved": None, "final": "",
}

print("=== Linear run ===")
linear_result = linear_graph.invoke(initial_state)
print("\nFinal state after all 4 nodes:")
for k, v in linear_result.items():
    print(f"  {k}: {v}")


=== Linear run ===


d:\internship\weak2\day3\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[plan] I will look up Laptop A and Laptop B to compare their prices, specifications, and overall value.
[retrieve] Laptop A: $650 (8GB RAM, 256GB SSD, 14-inch)
Laptop B: $950 (16GB RAM, 512GB SSD, 15-inch)


d:\internship\weak2\day3\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[generate] For a budget-conscious client, Laptop A is the better choice at $650 compared to Laptop B's higher price of $950. While Laptop B offers superior specifications, Laptop A provides essential performance at a much more affordable price point, making it ideal for cost-saving needs.


d:\internship\weak2\day3\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[critique] score=100 feedback=The draft successfully mentions both laptop prices and clearly explains why Laptop A is the better choice for a budget-conscious client.

Final state after all 4 nodes:
  question: Which laptop should a budget-conscious client buy, Laptop A or Laptop B?
  plan: I will look up Laptop A and Laptop B to compare their prices, specifications, and overall value.
  retrieved: Laptop A: $650 (8GB RAM, 256GB SSD, 14-inch)
Laptop B: $950 (16GB RAM, 512GB SSD, 15-inch)
  draft: For a budget-conscious client, Laptop A is the better choice at $650 compared to Laptop B's higher price of $950. While Laptop B offers superior specifications, Laptop A provides essential performance at a much more affordable price point, making it ideal for cost-saving needs.
  critique_feedback: The draft successfully mentions both laptop prices and clearly explains why Laptop A is the better choice for a budget-conscious client.
  score: 100
  retries: 0
  max_retries: 2
  approved: None
 

## Task 3: Add Conditional Edges & Cycles

Add a `route_critique` conditional edge: if the critique score is below 80 **and** retries
haven't hit `max_retries`, loop back to `generate` via a `revise` node. Otherwise, move on to
the (interrupt-gated) `send_email` node.


In [42]:
def revise_node(state: RecommendationState) -> dict:
    """Increments the retry counter before looping back to generate. Logs each pass."""
    new_retries = state["retries"] + 1
    print(f"[revise] retry #{new_retries} — looping back to generate with feedback: {state['critique_feedback']}")
    return {"retries": new_retries}


def route_critique(state: RecommendationState) -> str:
    """Conditional edge function: decide whether to loop back or move forward."""
    if state["score"] >= 80:
        return "send_email"
    if state["retries"] >= state["max_retries"]:
        print("[route_critique] max retries reached — moving on despite low score")
        return "send_email"
    return "revise"


def send_email_node(state: RecommendationState) -> dict:
    """The 'risky' action this workflow gates behind human approval."""
    return {"final": f"EMAIL SENT to client: {state['draft']}"}


**Why this loop-back is awkward in `AgentExecutor` but natural in LangGraph:** `AgentExecutor`
only exposes one implicit loop — reason, call a tool, observe, repeat — driven entirely by what the
model itself decides to do next; there's no way to say "always re-run this specific step if a
separate quality check fails" without hiding that logic inside the prompt or a custom loop around
`AgentExecutor` itself. In LangGraph, `generate` and `critique` are separate named nodes and
`route_critique` is ordinary Python that inspects `state["score"]`, so the retry policy is explicit,
inspectable code rather than something the model has to be talked into doing correctly.


In [43]:
cyclical_builder = StateGraph(RecommendationState)
cyclical_builder.add_node("plan", plan_node)
cyclical_builder.add_node("retrieve", retrieve_node)
cyclical_builder.add_node("generate", generate_node)
cyclical_builder.add_node("critique", critique_node)
cyclical_builder.add_node("revise", revise_node)
cyclical_builder.add_node("send_email", send_email_node)

cyclical_builder.add_edge(START, "plan")
cyclical_builder.add_edge("plan", "retrieve")
cyclical_builder.add_edge("retrieve", "generate")
cyclical_builder.add_edge("generate", "critique")
cyclical_builder.add_conditional_edges(
    "critique", route_critique, {"send_email": "send_email", "revise": "revise"}
)
cyclical_builder.add_edge("revise", "generate")
cyclical_builder.add_edge("send_email", END)

# Task 4/5 wiring: a checkpointer for persistence, and interrupt_before to pause
# for human approval right before the risky "send_email" action.
checkpointer = InMemorySaver()
graph = cyclical_builder.compile(checkpointer=checkpointer, interrupt_before=["send_email"])
print("Graph compiled with conditional edges, a self-correction loop, and a human approval gate.")


Graph compiled with conditional edges, a self-correction loop, and a human approval gate.


## Task 4: Human-in-the-Loop & Interrupts

`interrupt_before=["send_email"]` (set at compile time above) makes the graph pause right before
`send_email` runs, saving the full state to the checkpointer. The app then decides — based on a
real human's input — whether to resume normally (approve) or skip the node with a cancellation
(reject).


In [44]:
config_approve = {"configurable": {"thread_id": "client-1-approve"}}

print("=== Run until the approval gate ===")
paused_state = graph.invoke(initial_state, config_approve)

pending = graph.get_state(config_approve)
print("\nPaused. Next node waiting to run:", pending.next)
print("Draft awaiting approval:\n", paused_state["draft"])
print("Quality score:", paused_state["score"], " | retries used:", paused_state["retries"])


=== Run until the approval gate ===


d:\internship\weak2\day3\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[plan] To answer this, I will look up the specifications, pricing, and current reviews for Laptop A and Laptop B to compare their value.
[retrieve] Laptop A: $650 (8GB RAM, 256GB SSD, 14-inch)
Laptop B: $950 (16GB RAM, 512GB SSD, 15-inch)


d:\internship\weak2\day3\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[generate] For a budget-conscious client, **Laptop A** is the recommended choice at $650, compared to Laptop B's higher price of $950. While Laptop B offers better performance specs, Laptop A provides essential everyday functionality at a significantly lower cost, making it the ideal option for saving money.


d:\internship\weak2\day3\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[critique] score=100 feedback=The draft effectively mentions both prices and clearly explains why Laptop A is the better choice for a budget-conscious client.

Paused. Next node waiting to run: ('send_email',)
Draft awaiting approval:
 For a budget-conscious client, **Laptop A** is the recommended choice at $650, compared to Laptop B's higher price of $950. While Laptop B offers better performance specs, Laptop A provides essential everyday functionality at a significantly lower cost, making it the ideal option for saving money.
Quality score: 100  | retries used: 0


In [45]:
# --- Simulate a human APPROVING the send ---
human_decision = "approve"  # in a real app this comes from a UI/CLI prompt

if human_decision == "approve":
    result = graph.invoke(None, config_approve)  # resumes from where it paused
    print("\nRESUMED (approved):", result["final"])



RESUMED (approved): EMAIL SENT to client: For a budget-conscious client, **Laptop A** is the recommended choice at $650, compared to Laptop B's higher price of $950. While Laptop B offers better performance specs, Laptop A provides essential everyday functionality at a significantly lower cost, making it the ideal option for saving money.


In [46]:
# --- Simulate a human REJECTING the send, on a fresh thread ---
config_reject = {"configurable": {"thread_id": "client-2-reject"}}
graph.invoke(initial_state, config_reject)
print("Paused for approval on thread 'client-2-reject'.")

# Reject: skip send_email entirely by injecting a cancellation as if send_email
# had run (as_node="send_email"), so the graph can still proceed to END cleanly.
graph.update_state(config_reject, {"final": "CANCELLED - human rejected the send"}, as_node="send_email")
rejected_result = graph.invoke(None, config_reject)
print("RESUMED (rejected):", rejected_result["final"])


d:\internship\weak2\day3\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[plan] I will look up Laptop A and Laptop B to compare their specifications, pricing, and overall value for a budget-conscious buyer.
[retrieve] Laptop A: $650 (8GB RAM, 256GB SSD, 14-inch)
Laptop B: $950 (16GB RAM, 512GB SSD, 15-inch)


d:\internship\weak2\day3\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[generate] For a budget-conscious client, Laptop A is the better choice at $650 compared to Laptop B at $950. While Laptop B offers higher specs, Laptop A provides essential performance for everyday tasks at a much more affordable price point, saving you $300.


d:\internship\weak2\day3\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[critique] score=100 feedback=The draft successfully includes both prices and provides a clear justification tailored to a budget-conscious client.
Paused for approval on thread 'client-2-reject'.
RESUMED (rejected): CANCELLED - human rejected the send


### When should a real product require human-in-the-loop vs. full autonomy?

Human approval earns its cost when a single action is **hard or costly to undo** and **wrong
often enough to matter** — sending an email to a real client, making a purchase, deleting data,
or anything with legal/financial/reputational consequences. Full autonomy is reasonable when
actions are **cheap to reverse or purely informational** — looking something up, drafting text
nobody sees yet, or internal calculations — because the cost of a mistake is low and the friction
of approval slows the workflow down for no real safety benefit. In between, a common pattern is
to let low-risk paths run autonomously (as `generate`/`critique`/`revise` do here) and gate only
the one genuinely risky step (`send_email`) behind a human — which is exactly what this graph does.


## Task 5: Persistence & Debugging

### Resuming a paused conversation

Because `checkpointer=InMemorySaver()` was passed at compile time, the graph's state for
`thread_id="client-1-approve"` persisted across the two separate `.invoke()` calls above — the
second call didn't need the original input again, only the same `config`. (`InMemorySaver` only
persists for the life of this Python process; swapping in a `SqliteSaver` or `PostgresSaver` would
make it survive an actual kernel restart, with no other code changes.)


In [47]:
# Proof that persistence is thread-scoped: two different thread_ids stayed
# completely independent even though they used the same compiled graph.
print("client-1-approve final state:", graph.get_state(config_approve).values["final"])
print("client-2-reject final state:", graph.get_state(config_reject).values["final"])


client-1-approve final state: EMAIL SENT to client: For a budget-conscious client, **Laptop A** is the recommended choice at $650, compared to Laptop B's higher price of $950. While Laptop B offers better performance specs, Laptop A provides essential everyday functionality at a significantly lower cost, making it the ideal option for saving money.
client-2-reject final state: CANCELLED - human rejected the send


### Time-travel / state history for debugging

In [48]:
history = list(graph.get_state_history(config_approve))
print(f"{len(history)} checkpoints recorded for thread 'client-1-approve':\n")
for snap in reversed(history):  # reversed = chronological order
    print(f"- next={snap.next!s:<16} retries={snap.values.get('retries')} score={snap.values.get('score')}")


7 checkpoints recorded for thread 'client-1-approve':

- next=('__start__',)   retries=None score=None
- next=('plan',)        retries=0 score=0
- next=('retrieve',)    retries=0 score=0
- next=('generate',)    retries=0 score=0
- next=('critique',)    retries=0 score=0
- next=('send_email',)  retries=0 score=100
- next=()               retries=0 score=100


In [49]:
# Replay/debug from an earlier checkpoint: find the snapshot taken right after
# the FIRST "generate" node ran, and branch a new run from there by re-invoking
# the graph with that snapshot's config instead of the latest one.
generate_checkpoints = [s for s in history if s.next == ("critique",)]
first_generate_snapshot = generate_checkpoints[-1]  # earliest one, chronologically

print("Replaying from the checkpoint right after the first 'generate' ran:")
print("Draft at that point:", first_generate_snapshot.values["draft"][:120], "...")

replayed = graph.invoke(None, config=first_generate_snapshot.config)
print("\nRe-running forward from that checkpoint reached:", replayed.get("final") or "(paused again for approval)")


Replaying from the checkpoint right after the first 'generate' ran:
Draft at that point: For a budget-conscious client, **Laptop A** is the recommended choice at $650, compared to Laptop B's higher price of $9 ...


d:\internship\weak2\day3\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[critique] score=100 feedback=The draft effectively includes both laptop prices and clearly explains why Laptop A is the better choice for a budget-conscious client.

Re-running forward from that checkpoint reached: (paused again for approval)


**`AgentExecutor` vs. LangGraph — when to reach for each:**

Use `AgentExecutor` (or Day 2's `create_agent`) when the task is a single, mostly-linear
reasoning loop with tools — a chatbot answering questions, a lookup-and-summarize task — where
"call tools until you have an answer" is genuinely the whole shape of the problem, and you don't
need to pause mid-run or replay history. Reach for LangGraph once the workflow needs **explicit
branching, a bounded retry/self-correction loop, a mandatory pause for human approval, or
persistence/replay across sessions** — anything where you'd otherwise have to hide that control
flow inside prompt engineering or a hand-written wrapper loop around `AgentExecutor`. In short:
`AgentExecutor`/`create_agent` for "reason your way to an answer with tools"; LangGraph for "run
a specific multi-step process, possibly with checkpoints, retries, and a human in the loop."
